# risk_budget_downside_vol_target_v1

Notebook-contained diagnostic experiment for the current SOXL overfit / drawdown target loop.

## Governance

- Direction: `execution`
- Changed layer: `execution` / position sizing post-processing
- Role: `diagnose` and `ablate`
- Canonical baseline: `risk_budget_usage_v1_actor_parity` daily path, because `selection_overfit_audit_v1` identified it as the lowest-MaxDD strict daily artifact that still had positive excess over buy-and-hold.
- Direct comparison baselines: unmodified `risk_budget_usage_v1_actor_parity`; prior conservative frontier from `runtime/model_selection/card/selection_overfit_audit_v1`.
- Target window: strict OOS full daily path plus fixed slices `test_2020`, `test_2022`, `test_2023_2024`.
- Acceptance criteria: strict-OOS `relative_return >= +30pp` and `max_dd <= 15%`, with no obvious fixed-slice collapse.
- Expected trade-off: lower MaxDD may give up bull participation; the run is only useful if the frontier crosses 15% while keeping at least +30pp excess.
- Failure mode: volatility targeting mechanically reduces exposure after large moves and removes too much positive convexity from SOXL rebounds.
- Closest historical success: `risk_budget_usage_v1_actor_parity` is already low drawdown relative to other artifacts.
- Closest historical failure: simple scaling / equity drawdown control in `selection_overfit_audit_v1` could not get below 15% while preserving +30pp excess.
- Why this is not a duplicate: this run tests downside-volatility scaling and a two-stage downside-vol plus equity-DD cap, not static scale/floor or raw equity-DD control alone.
- Comparison-table row key: `risk_budget_downside_vol_target_v1`.

## External References

- Harvey et al., *The Impact of Volatility Targeting*: volatility targeting can reduce risk, especially for assets with negative return/volatility relation. https://papers.ssrn.com/sol3/papers.cfm?abstract_id=3175538
- Moreira and Muir, *Volatility-Managed Portfolios*: scale exposure inversely with recent variance; performance depends on asset class and timing. https://academic.oup.com/jof/article/72/4/1611/7789645
- Smoothing volatility targeting methods: turnover-aware volatility targeting variants. https://arxiv.org/abs/2212.07288
- Lim et al., *Enhancing Time-Series Momentum Strategies Using Deep Neural Networks*: separates trend signal from position sizing / risk control. https://arxiv.org/abs/1904.04912
- Qlib: keep the research loop inside the existing qlib-style harness. https://github.com/microsoft/qlib


In [1]:
from __future__ import annotations

from itertools import product
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "runtime").exists():
            return candidate
    raise RuntimeError(f"could not locate repo root from {start}")


ROOT = find_repo_root()
CARD = ROOT / "runtime/execution/card/risk_budget_downside_vol_target_v1"
CARD.mkdir(parents=True, exist_ok=True)
BASE_DAILY_PATH = ROOT / "runtime/model/card/risk_budget_usage_v1/risk_budget_usage_v1_actor_parity_soxl_daily.csv"
BASELINE_LABEL = "risk_budget_usage_v1_actor_parity"
EXPERIMENT = "risk_budget_downside_vol_target_v1"

df = pd.read_csv(BASE_DAILY_PATH)
df["datetime"] = pd.to_datetime(df["datetime"])
df = df.sort_values("datetime").reset_index(drop=True)
price_returns = df["price"].pct_change()

WINDOWS = {
    "test_2020": ("2020-01-01", "2020-12-31"),
    "test_2022": ("2022-01-01", "2022-12-31"),
    "test_2023_2024": ("2023-01-01", "2024-12-31"),
}

print(BASE_DAILY_PATH.relative_to(ROOT), df.shape, df["datetime"].min().date(), df["datetime"].max().date())

runtime/model/card/risk_budget_usage_v1/risk_budget_usage_v1_actor_parity_soxl_daily.csv (1569, 8) 2020-01-02 2026-03-31


In [2]:
def metrics(frame: pd.DataFrame, *, start: str | None = None, end: str | None = None) -> dict:
    z = frame.copy()
    if start is not None:
        z = z[z["datetime"] >= pd.Timestamp(start)]
    if end is not None:
        z = z[z["datetime"] <= pd.Timestamp(end)]
    prices = z["price"].to_numpy(dtype=float)
    positions = z["position"].to_numpy(dtype=float)
    if len(z) < 2:
        raise ValueError("not enough rows for metrics")
    returns = np.diff(prices) / prices[:-1]
    strategy_returns = positions[:-1] * returns
    nav = np.concatenate([[1.0], np.cumprod(1.0 + strategy_returns)])
    buyhold = np.concatenate([[1.0], np.cumprod(1.0 + returns)])
    peak = np.maximum.accumulate(nav)
    drawdown = (nav - peak) / peak
    stdev = float(np.std(strategy_returns))
    return {
        "rows": int(len(z)),
        "start": z["datetime"].iloc[0].date().isoformat(),
        "end": z["datetime"].iloc[-1].date().isoformat(),
        "return": float(nav[-1] - 1.0) * 100.0,
        "sharpe": float(np.mean(strategy_returns) / stdev * np.sqrt(252)) if stdev > 0 else 0.0,
        "max_dd": abs(float(np.min(drawdown))) * 100.0,
        "buy_and_hold_return": float(buyhold[-1] - 1.0) * 100.0,
        "relative_return": float(nav[-1] - buyhold[-1]) * 100.0,
        "avg_pos": float(np.mean(positions)),
        "trades": int(np.sum(np.abs(np.diff(positions)) > 0.001)),
    }


def slice_metric_row(frame: pd.DataFrame, label: str) -> dict:
    row = {"label": label, **metrics(frame)}
    for name, (start, end) in WINDOWS.items():
        m = metrics(frame, start=start, end=end)
        for key in ["return", "sharpe", "max_dd", "relative_return", "avg_pos", "trades"]:
            row[f"{name}_{key}"] = m[key]
    row["target_pass"] = bool(row["relative_return"] >= 30.0 and row["max_dd"] <= 15.0)
    return row


def attach_nav(frame: pd.DataFrame) -> pd.DataFrame:
    z = frame.copy()
    prices = z["price"].to_numpy(dtype=float)
    positions = z["position"].to_numpy(dtype=float)
    returns = np.diff(prices) / prices[:-1]
    strategy_nav = np.concatenate([[1.0], np.cumprod(1.0 + positions[:-1] * returns)])
    buyhold_nav = np.concatenate([[1.0], np.cumprod(1.0 + returns)])
    z["strategy_nav"] = strategy_nav * 1_000_000.0
    z["buyhold_nav"] = buyhold_nav * 1_000_000.0
    z["excess"] = z["strategy_nav"] - z["buyhold_nav"]
    z["excess_pct"] = z["excess"] / z["buyhold_nav"] * 100.0
    actions = ["HOLD"]
    for delta in np.diff(positions):
        actions.append("BUY" if delta > 0.001 else "SELL" if delta < -0.001 else "HOLD")
    z["action"] = actions
    leading = ["datetime", "strategy_nav", "price", "buyhold_nav", "position", "excess", "excess_pct", "action"]
    return z[leading + [c for c in z.columns if c not in leading]]

baseline_row = slice_metric_row(df[["datetime", "price", "position"]], BASELINE_LABEL)
baseline_metrics = pd.DataFrame([baseline_row])
display(baseline_metrics.T.loc[["return", "sharpe", "max_dd", "relative_return", "avg_pos", "trades", "target_pass"]])

,0
return,233.856985
sharpe,1.028952
max_dd,21.375029
relative_return,87.710263
avg_pos,0.180393
trades,1320
target_pass,False


In [3]:
def downside_vol_scale(
    frame: pd.DataFrame,
    *,
    window: int,
    target_vol: float,
    max_scale: float,
    band: float,
    trend_filter: bool,
) -> pd.DataFrame:
    out = frame.copy()
    returns = out["price"].pct_change()
    downside_risk = returns.clip(upper=0).rolling(window, min_periods=max(5, window // 2)).std() * np.sqrt(252)
    scale = (target_vol / downside_risk).replace([np.inf, -np.inf], np.nan).clip(0.0, max_scale).fillna(1.0)
    if trend_filter:
        ma200 = out["price"].rolling(200, min_periods=50).mean()
        mom21 = out["price"] / out["price"].shift(21) - 1.0
        scale = pd.Series(
            np.where((out["price"] > ma200) & (mom21 > 0.0), scale, np.minimum(scale, 1.0)),
            index=out.index,
        ).fillna(1.0)
    desired = (out["position"] * scale).clip(0.0, 1.0).to_numpy(dtype=float)
    positions: list[float] = []
    current = 0.0
    for target in desired:
        target = float(target)
        if abs(target - current) >= band:
            current = target
        positions.append(current)
    out["base_position"] = out["position"]
    out["position"] = positions
    out["downside_risk_estimate"] = downside_risk
    out["vol_scale"] = scale
    return out


def target_gap(row: pd.Series) -> float:
    return max(0.0, row["max_dd"] - 15.0) * 4.0 + max(0.0, 30.0 - row["relative_return"])

vol_rows = []
vol_candidates: dict[str, pd.DataFrame] = {}
for window, target_vol, max_scale, band, trend_filter in product(
    [5, 10, 21, 42, 63],
    [0.30, 0.35, 0.40, 0.45, 0.50, 0.60, 0.80],
    [1.0, 1.5, 2.0, 3.0, 4.0],
    [0.01, 0.02, 0.04, 0.06],
    [False, True],
):
    candidate = downside_vol_scale(
        df, window=window, target_vol=target_vol, max_scale=max_scale, band=band, trend_filter=trend_filter
    )
    label = f"vol_w{window}_tv{target_vol:.2f}_mx{max_scale:.1f}_b{band:.2f}_trend{int(trend_filter)}"
    row = slice_metric_row(candidate, label)
    row.update({
        "window": window,
        "target_vol": target_vol,
        "max_scale": max_scale,
        "band": band,
        "trend_filter": trend_filter,
        "target_gap": target_gap(pd.Series(row)),
    })
    vol_rows.append(row)
    vol_candidates[label] = candidate

vol_screen = pd.DataFrame(vol_rows).sort_values(["target_pass", "target_gap", "max_dd"], ascending=[False, True, True])
vol_screen.to_csv(CARD / "downside_vol_target_screen.csv", index=False)

vol_target_pass = vol_screen[vol_screen["target_pass"]]
vol_relative_frontier = vol_screen[vol_screen["relative_return"] >= 30.0].sort_values(["max_dd", "target_gap"])
vol_low_dd_frontier = vol_screen.sort_values(["max_dd", "target_gap"])

print("vol screen rows", len(vol_screen), "target_pass", len(vol_target_pass))
display(vol_relative_frontier.head(15)[["label", "window", "target_vol", "max_scale", "band", "trend_filter", "return", "sharpe", "max_dd", "relative_return", "test_2020_return", "test_2020_max_dd", "test_2022_return", "test_2022_max_dd", "test_2023_2024_return", "test_2023_2024_max_dd", "avg_pos", "trades", "target_pass"]])

vol screen rows 1400 target_pass 0


,label,window,target_vol,max_scale,band,trend_filter,return,sharpe,max_dd,relative_return,test_2020_return,test_2020_max_dd,test_2022_return,test_2022_max_dd,test_2023_2024_return,test_2023_2024_max_dd,avg_pos,trades,target_pass
355,vol_w10_tv0.35_mx4.0_b0.02_trend1,10,0.35,4.0,0.02,True,177.479796,0.960142,16.277862,31.333073,19.285428,15.489142,-7.628252,11.189810,32.578923,15.994587,0.163345,401,False
351,vol_w10_tv0.35_mx3.0_b0.06_trend1,10,0.35,3.0,0.06,True,176.742340,0.960783,16.545323,30.595618,22.513154,15.334514,-8.101362,11.894905,34.868278,15.797577,0.165473,185,False
660,vol_w21_tv0.40_mx2.0_b0.04_trend0,21,0.40,2.0,0.04,False,180.305029,1.066259,16.759491,34.158307,23.095886,12.493112,-6.590862,12.924083,38.614732,16.759491,0.156285,174,False
676,vol_w21_tv0.40_mx4.0_b0.04_trend0,21,0.40,4.0,0.04,False,188.996446,1.065703,16.759491,42.849724,23.095886,12.493112,-6.590862,12.924083,38.614732,16.759491,0.157944,176,False
652,vol_w21_tv0.40_mx1.5_b0.04_trend0,21,0.40,1.5,0.04,False,176.719699,1.075604,16.759491,30.572976,23.329128,12.493112,-6.590862,12.924083,37.701051,16.759491,0.153683,169,False
668,vol_w21_tv0.40_mx3.0_b0.04_trend0,21,0.40,3.0,0.04,False,188.390926,1.074626,16.759491,42.244203,23.095886,12.493112,-6.590862,12.924083,38.614732,16.759491,0.157540,176,False
666,vol_w21_tv0.40_mx3.0_b0.02_trend0,21,0.40,3.0,0.02,False,178.632071,1.045008,16.804876,32.485348,21.476421,12.538519,-5.479242,11.659117,40.410335,16.804876,0.156779,305,False
674,vol_w21_tv0.40_mx4.0_b0.02_trend0,21,0.40,4.0,0.02,False,179.217101,1.036382,16.804876,33.070378,21.476421,12.538519,-5.479242,11.659117,40.410335,16.804876,0.157183,305,False
359,vol_w10_tv0.35_mx4.0_b0.06_trend1,10,0.35,4.0,0.06,True,183.367624,0.966258,16.880556,37.220902,23.152925,15.334514,-8.101362,11.894905,34.449023,15.797577,0.167050,189,False
349,vol_w10_tv0.35_mx3.0_b0.04_trend1,10,0.35,3.0,0.04,True,187.475527,1.001848,16.976781,41.328804,20.517501,15.429875,-6.897245,11.098244,36.845148,15.296426,0.163221,246,False


In [4]:
def apply_equity_drawdown_cap(
    frame: pd.DataFrame,
    *,
    band: float,
    start_drawdown: float,
    full_drawdown: float,
    min_target: float,
) -> tuple[pd.DataFrame, int]:
    out = frame.copy()
    desired = out["position"].to_numpy(dtype=float)
    prices = out["price"].to_numpy(dtype=float)
    positions: list[float] = []
    current = 0.0
    nav = 1.0
    peak = 1.0
    previous_price: float | None = None
    active_days = 0
    equity_navs: list[float] = []
    drawdowns: list[float] = []
    caps: list[float] = []
    for idx, price in enumerate(prices):
        if previous_price is not None and previous_price > 0.0:
            nav *= 1.0 + current * (price / previous_price - 1.0)
        peak = max(peak, nav)
        drawdown = max(1.0 - nav / max(peak, 1e-12), 0.0)
        if drawdown < start_drawdown:
            cap = 1.0
        elif full_drawdown <= start_drawdown or drawdown >= full_drawdown:
            cap = min_target
        else:
            usage = (drawdown - start_drawdown) / max(full_drawdown - start_drawdown, 1e-12)
            cap = max(min_target, 1.0 - usage * (1.0 - min_target))
        target = min(float(desired[idx]), cap)
        if cap < 1.0:
            active_days += 1
        threshold = 0.0 if cap < 1.0 and target < current else band
        if abs(target - current) >= threshold:
            current = target
        positions.append(current)
        previous_price = price
        equity_navs.append(nav)
        drawdowns.append(drawdown)
        caps.append(cap)
    out["pre_dd_cap_position"] = out["position"]
    out["position"] = positions
    out["equity_nav"] = equity_navs
    out["equity_drawdown"] = drawdowns
    out["equity_drawdown_cap"] = caps
    return out, active_days

# Only run the second-stage cap over the nearest positive-excess volatility frontier.
frontier_labels = vol_relative_frontier.head(12)["label"].tolist()
combo_rows = []
combo_candidates: dict[str, pd.DataFrame] = {}
for label in frontier_labels:
    base_candidate = vol_candidates[label]
    base_row = vol_screen[vol_screen["label"] == label].iloc[0].to_dict()
    band = float(base_row["band"])
    for start_drawdown, full_drawdown, min_target in product(
        [0.08, 0.10, 0.12, 0.14],
        [0.13, 0.15, 0.17, 0.20],
        [0.00, 0.05, 0.10, 0.15, 0.20],
    ):
        if full_drawdown <= start_drawdown:
            continue
        candidate, active_days = apply_equity_drawdown_cap(
            base_candidate,
            band=band,
            start_drawdown=start_drawdown,
            full_drawdown=full_drawdown,
            min_target=min_target,
        )
        combo_label = f"{label}_dd{start_drawdown:.2f}_{full_drawdown:.2f}_min{min_target:.2f}"
        row = slice_metric_row(candidate, combo_label)
        row.update({
            "source_label": label,
            "start_drawdown": start_drawdown,
            "full_drawdown": full_drawdown,
            "min_target": min_target,
            "active_days": active_days,
            "target_gap": target_gap(pd.Series(row)),
        })
        combo_rows.append(row)
        combo_candidates[combo_label] = candidate

combo_screen = pd.DataFrame(combo_rows).sort_values(["target_pass", "target_gap", "max_dd"], ascending=[False, True, True])
combo_screen.to_csv(CARD / "vol_plus_equity_dd_cap_screen.csv", index=False)
combo_target_pass = combo_screen[combo_screen["target_pass"]]
combo_relative_frontier = combo_screen[combo_screen["relative_return"] >= 30.0].sort_values(["max_dd", "target_gap"])
combo_low_dd_frontier = combo_screen.sort_values(["max_dd", "target_gap"])

print("combo screen rows", len(combo_screen), "target_pass", len(combo_target_pass))
display(combo_relative_frontier.head(15)[["label", "source_label", "start_drawdown", "full_drawdown", "min_target", "return", "sharpe", "max_dd", "relative_return", "test_2020_return", "test_2020_max_dd", "test_2022_return", "test_2022_max_dd", "test_2023_2024_return", "test_2023_2024_max_dd", "active_days", "avg_pos", "trades", "target_pass"]])
display(combo_low_dd_frontier.head(10)[["label", "source_label", "start_drawdown", "full_drawdown", "min_target", "return", "sharpe", "max_dd", "relative_return", "active_days", "avg_pos", "trades", "target_pass"]])

combo screen rows 900 target_pass 0


,label,source_label,start_drawdown,full_drawdown,min_target,return,sharpe,max_dd,relative_return,test_2020_return,test_2020_max_dd,test_2022_return,test_2022_max_dd,test_2023_2024_return,test_2023_2024_max_dd,active_days,avg_pos,trades,target_pass
52,vol_w10_tv0.35_mx4.0_b0.02_trend1_dd0.12_0.17_...,vol_w10_tv0.35_mx4.0_b0.02_trend1,0.12,0.17,0.10,177.206488,0.959340,16.277862,31.059765,19.285428,15.489142,-7.628252,11.18981,32.448337,16.077329,71,0.163318,402,False
15,vol_w10_tv0.35_mx4.0_b0.02_trend1_dd0.08_0.20_...,vol_w10_tv0.35_mx4.0_b0.02_trend1,0.08,0.20,0.00,177.361269,0.959856,16.277862,31.214547,19.411791,15.399617,-7.628252,11.18981,32.382054,16.119328,247,0.163240,402,False
34,vol_w10_tv0.35_mx4.0_b0.02_trend1_dd0.10_0.17_...,vol_w10_tv0.35_mx4.0_b0.02_trend1,0.10,0.17,0.20,177.167031,0.959238,16.277862,31.020308,19.314652,15.468437,-7.628252,11.18981,32.397048,16.109827,107,0.163293,402,False
18,vol_w10_tv0.35_mx4.0_b0.02_trend1_dd0.08_0.20_...,vol_w10_tv0.35_mx4.0_b0.02_trend1,0.08,0.20,0.15,177.598455,0.960516,16.277862,31.451732,19.336438,15.453003,-7.628252,11.18981,32.578923,15.994587,247,0.163319,401,False
14,vol_w10_tv0.35_mx4.0_b0.02_trend1_dd0.08_0.17_...,vol_w10_tv0.35_mx4.0_b0.02_trend1,0.08,0.17,0.20,176.953329,0.958665,16.277862,30.806606,19.445282,15.375889,-7.628252,11.18981,32.150285,16.266183,247,0.163173,402,False
19,vol_w10_tv0.35_mx4.0_b0.02_trend1_dd0.08_0.20_...,vol_w10_tv0.35_mx4.0_b0.02_trend1,0.08,0.20,0.20,177.540026,0.960332,16.277862,31.393303,19.311320,15.470798,-7.628252,11.18981,32.578923,15.994587,247,0.163331,401,False
33,vol_w10_tv0.35_mx4.0_b0.02_trend1_dd0.10_0.17_...,vol_w10_tv0.35_mx4.0_b0.02_trend1,0.10,0.17,0.15,176.952961,0.958617,16.277862,30.806238,19.339978,15.450494,-7.628252,11.18981,32.266716,16.192409,107,0.163253,402,False
35,vol_w10_tv0.35_mx4.0_b0.02_trend1_dd0.10_0.20_...,vol_w10_tv0.35_mx4.0_b0.02_trend1,0.10,0.20,0.00,177.479796,0.960142,16.277862,31.333073,19.285428,15.489142,-7.628252,11.18981,32.578923,15.994587,107,0.163345,401,False
36,vol_w10_tv0.35_mx4.0_b0.02_trend1_dd0.10_0.20_...,vol_w10_tv0.35_mx4.0_b0.02_trend1,0.10,0.20,0.05,177.479796,0.960142,16.277862,31.333073,19.285428,15.489142,-7.628252,11.18981,32.578923,15.994587,107,0.163345,401,False
37,vol_w10_tv0.35_mx4.0_b0.02_trend1_dd0.10_0.20_...,vol_w10_tv0.35_mx4.0_b0.02_trend1,0.10,0.20,0.10,177.479796,0.960142,16.277862,31.333073,19.285428,15.489142,-7.628252,11.18981,32.578923,15.994587,107,0.163345,401,False


,label,source_label,start_drawdown,full_drawdown,min_target,return,sharpe,max_dd,relative_return,active_days,avg_pos,trades,target_pass
450,vol_w21_tv0.40_mx3.0_b0.02_trend0_dd0.08_0.13_...,vol_w21_tv0.40_mx3.0_b0.02_trend0,0.08,0.13,0.0,57.698422,0.617777,13.000073,-88.448301,698,0.097001,186,False
525,vol_w21_tv0.40_mx4.0_b0.02_trend0_dd0.08_0.13_...,vol_w21_tv0.40_mx4.0_b0.02_trend0,0.08,0.13,0.0,57.627896,0.614369,13.000073,-88.518827,698,0.097166,186,False
395,vol_w21_tv0.40_mx3.0_b0.04_trend0_dd0.10_0.13_...,vol_w21_tv0.40_mx3.0_b0.04_trend0,0.10,0.13,0.0,36.541816,0.505787,13.017879,-109.604906,883,0.069752,79,False
320,vol_w21_tv0.40_mx1.5_b0.04_trend0_dd0.10_0.13_...,vol_w21_tv0.40_mx1.5_b0.04_trend0,0.10,0.13,0.0,36.537495,0.520466,13.017879,-109.609228,883,0.068265,75,False
245,vol_w21_tv0.40_mx4.0_b0.04_trend0_dd0.10_0.13_...,vol_w21_tv0.40_mx4.0_b0.04_trend0,0.10,0.13,0.0,36.480752,0.501834,13.017879,-109.665971,883,0.069917,79,False
170,vol_w21_tv0.40_mx2.0_b0.04_trend0_dd0.10_0.13_...,vol_w21_tv0.40_mx2.0_b0.04_trend0,0.10,0.13,0.0,35.915017,0.502964,13.017879,-110.231706,883,0.069295,78,False
225,vol_w21_tv0.40_mx4.0_b0.04_trend0_dd0.08_0.13_...,vol_w21_tv0.40_mx4.0_b0.04_trend0,0.08,0.13,0.0,34.283031,0.479913,13.032261,-111.863692,915,0.070256,90,False
150,vol_w21_tv0.40_mx2.0_b0.04_trend0_dd0.08_0.13_...,vol_w21_tv0.40_mx2.0_b0.04_trend0,0.08,0.13,0.0,33.726406,0.480676,13.032261,-112.420317,915,0.069634,89,False
300,vol_w21_tv0.40_mx1.5_b0.04_trend0_dd0.08_0.13_...,vol_w21_tv0.40_mx1.5_b0.04_trend0,0.08,0.13,0.0,34.338860,0.497686,13.032261,-111.807863,914,0.068604,86,False
375,vol_w21_tv0.40_mx3.0_b0.04_trend0_dd0.08_0.13_...,vol_w21_tv0.40_mx3.0_b0.04_trend0,0.08,0.13,0.0,34.343112,0.483709,13.032261,-111.803611,915,0.070091,90,False


In [5]:
summary_rows = [baseline_row]
if len(vol_relative_frontier):
    best_vol_label = vol_relative_frontier.iloc[0]["label"]
else:
    best_vol_label = vol_screen.iloc[0]["label"]
best_vol = vol_candidates[best_vol_label]
best_vol_row = slice_metric_row(best_vol, f"{EXPERIMENT}_best_downside_vol")
summary_rows.append(best_vol_row)

if len(combo_relative_frontier):
    best_combo_label = combo_relative_frontier.iloc[0]["label"]
else:
    best_combo_label = combo_screen.iloc[0]["label"]
best_combo = combo_candidates[best_combo_label]
best_combo_row = slice_metric_row(best_combo, f"{EXPERIMENT}_best_combo")
summary_rows.append(best_combo_row)

summary = pd.DataFrame(summary_rows)
summary.to_csv(CARD / "selection_summary.csv", index=False)
attach_nav(best_vol).to_csv(CARD / "risk_budget_downside_vol_target_v1_best_downside_vol_soxl_daily.csv", index=False)
attach_nav(best_combo).to_csv(CARD / "risk_budget_downside_vol_target_v1_best_combo_soxl_daily.csv", index=False)

selection = {
    "experiment": EXPERIMENT,
    "baseline_path": str(BASE_DAILY_PATH),
    "baseline": baseline_row,
    "vol_screen_rows": int(len(vol_screen)),
    "vol_target_pass_rows": int(len(vol_target_pass)),
    "combo_screen_rows": int(len(combo_screen)),
    "combo_target_pass_rows": int(len(combo_target_pass)),
    "best_downside_vol_label": best_vol_label,
    "best_combo_label": best_combo_label,
    "best_downside_vol": best_vol_row,
    "best_combo": best_combo_row,
    "decision": "diagnostic_keep_no_promotion",
}
(CARD / "selection.json").write_text(json.dumps(selection, indent=2), encoding="utf-8")

display(summary[["label", "return", "sharpe", "max_dd", "relative_return", "test_2020_return", "test_2020_max_dd", "test_2022_return", "test_2022_max_dd", "test_2023_2024_return", "test_2023_2024_max_dd", "avg_pos", "trades", "target_pass"]])

,label,return,sharpe,max_dd,relative_return,test_2020_return,test_2020_max_dd,test_2022_return,test_2022_max_dd,test_2023_2024_return,test_2023_2024_max_dd,avg_pos,trades,target_pass
0,risk_budget_usage_v1_actor_parity,233.856985,1.028952,21.375029,87.710263,38.452544,17.626678,-8.872797,18.869923,35.750847,19.462175,0.180393,1320,False
1,risk_budget_downside_vol_target_v1_best_downsi...,177.479796,0.960142,16.277862,31.333073,19.285428,15.489142,-7.628252,11.189810,32.578923,15.994587,0.163345,401,False
2,risk_budget_downside_vol_target_v1_best_combo,177.206488,0.959340,16.277862,31.059765,19.285428,15.489142,-7.628252,11.189810,32.448337,16.077329,0.163318,402,False


In [6]:
vol_pass = int(len(vol_target_pass))
combo_pass = int(len(combo_target_pass))
closest_vol = vol_relative_frontier.iloc[0] if len(vol_relative_frontier) else vol_screen.iloc[0]
closest_combo = combo_relative_frontier.iloc[0] if len(combo_relative_frontier) else combo_screen.iloc[0]
lowest_combo = combo_low_dd_frontier.iloc[0]

readout = f"""
## Final Readout

- Baseline `{BASELINE_LABEL}`: return `{baseline_row['return']:.2f}%`, Sharpe `{baseline_row['sharpe']:.3f}`, MaxDD `{baseline_row['max_dd']:.2f}%`, relative return `{baseline_row['relative_return']:.2f}pp`.
- Downside-volatility screen rows: `{len(vol_screen)}`; target-pass rows: `{vol_pass}`.
- Best positive-excess downside-vol frontier: `{closest_vol['label']}` with return `{closest_vol['return']:.2f}%`, MaxDD `{closest_vol['max_dd']:.2f}%`, relative `{closest_vol['relative_return']:.2f}pp`, avg position `{closest_vol['avg_pos']:.3f}`.
- Vol plus equity-DD cap screen rows: `{len(combo_screen)}`; target-pass rows: `{combo_pass}`.
- Best positive-excess combo frontier: `{closest_combo['label']}` with return `{closest_combo['return']:.2f}%`, MaxDD `{closest_combo['max_dd']:.2f}%`, relative `{closest_combo['relative_return']:.2f}pp`.
- Lowest-DD combo overall: `{lowest_combo['label']}` with MaxDD `{lowest_combo['max_dd']:.2f}%` and relative `{lowest_combo['relative_return']:.2f}pp`.

Decision: diagnostic keep / no runtime promotion. Downside-vol targeting can bring the low-drawdown risk-budget family very close to the 15% MaxDD target, but the strict target frontier still does not cross both gates. Adding a second-stage account drawdown cap pushes MaxDD under 15% only by giving up too much buy-hold-relative return. This supports the next experiment being a state-aware re-risk / bull-participation recovery mechanism on top of downside-vol targeting, not another unconditional shrinkage pass.
"""
display(Markdown(readout))


## Final Readout

- Baseline `risk_budget_usage_v1_actor_parity`: return `233.86%`, Sharpe `1.029`, MaxDD `21.38%`, relative return `87.71pp`.
- Downside-volatility screen rows: `1400`; target-pass rows: `0`.
- Best positive-excess downside-vol frontier: `vol_w10_tv0.35_mx4.0_b0.02_trend1` with return `177.48%`, MaxDD `16.28%`, relative `31.33pp`, avg position `0.163`.
- Vol plus equity-DD cap screen rows: `900`; target-pass rows: `0`.
- Best positive-excess combo frontier: `vol_w10_tv0.35_mx4.0_b0.02_trend1_dd0.12_0.17_min0.10` with return `177.21%`, MaxDD `16.28%`, relative `31.06pp`.
- Lowest-DD combo overall: `vol_w21_tv0.40_mx3.0_b0.02_trend0_dd0.08_0.13_min0.00` with MaxDD `13.00%` and relative `-88.45pp`.

Decision: diagnostic keep / no runtime promotion. Downside-vol targeting can bring the low-drawdown risk-budget family very close to the 15% MaxDD target, but the strict target frontier still does not cross both gates. Adding a second-stage account drawdown cap pushes MaxDD under 15% only by giving up too much buy-hold-relative return. This supports the next experiment being a state-aware re-risk / bull-participation recovery mechanism on top of downside-vol targeting, not another unconditional shrinkage pass.
